# 1. Initializations

## 1.1 General imports

In [ ]:
### global
import logging
import joblib
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)
import itertools 

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline


## 1.2 Common imports

In [ ]:
import smartcheck.dataframe_common as dfc

## 1.3 Project Specific imports

In [ ]:
import smartcheck.dataframe_project_specific as dfps
import smartcheck.preprocessing_project_specific as pps
import smartcheck.modeling_project_specific as mps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    # "latitude",
    # "longitude",
    # "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    # "weather_code_wmo_code",
    # "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePreprocessingTransformer(timestamp_col="date_et_heure_de_comptage", for_sarimax=True)),
])

df_preproc = pipe_preproc.fit_transform(df_cpt)
if df_preproc is not None and isinstance(df_preproc, pd.DataFrame):
    df = df_preproc.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

## 2.2 Column Transformers

#### Categorical and Numerical

In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
# s_scaler = StandardScaler()
mm_scaler = MinMaxScaler()
ohe_enc = OneHotEncoder(
    # drop='first', # évites la multicolinéarité mais déclenche des warning (les catégories inconnues participent a renforcer la catégorie droppée...)
    handle_unknown='ignore'
)
tr_num_col = Pipeline(
    steps = [
        ('standardisation', mm_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = ColumnTransformer(
    transformers=[ 
        ('num_col_transf', tr_num_col, num_col),
        ('cat_col_transf', tr_cat_col, cat_col)
    ],
    remainder='drop',
)

#### Test de la pipeline column transformer

In [ ]:
pipe_columns = Pipeline(
    steps= [
        ('prep', tr_columns), 
    ]
)

In [ ]:
# Fit la pipeline
array_tr = pipe_columns.fit_transform(df)
# Si sparse matrix, convertir en dense
if hasattr(array_tr, "toarray"):
    array_tr = array_tr.toarray()  # type: ignore
# Récupération des noms de colonnes
num_col_names = num_col  # inchangés
# Récupérer l'encodeur entraîné depuis la pipeline
ohe_encoder_fitted = (
    pipe_columns.named_steps['prep']
    .named_transformers_['cat_col_transf']  
)
# Récupérer les noms de colonnes encodées
cat_col_names = ohe_encoder_fitted.get_feature_names_out(cat_col)
# Concaténation des noms
all_feature_names = np.concatenate([num_col_names, cat_col_names])
# Création du DataFrame
df_tr = pd.DataFrame(array_tr, columns=all_feature_names, index=df.index)

# 3. Regression modeling

In [ ]:
dict_compteurs = {
    ('Totem 73 boulevard de Sébastopol', 'S-N'): {
        "name": "Sebastopol S-N",
        "sub_range": range(1500, 5001),
        "order": (6, 1, 3),
        "seasonal_order": (1, 1, 1, 24),
        "use_exo": True,
    },
}
timestamp_col = "date_et_heure_de_comptage_local"
target_col = "comptage_horaire"

## 3.1 SARIMAX simple (sans exogène)

In [ ]:
def detect_datetime_columns(df: pd.DataFrame) -> list[str]:
    return [
        col for col in df.columns
        if pd.api.types.is_datetime64_any_dtype(df[col])
    ]

In [ ]:
def affichage_pacf_acf():
    # Affichage des graphes de PACF/ACF pour illustrer la pertinence des ordres définis sur les données de train et test
    def ajouter_lignes_saisonnieres(ax, seasonal_lags):
        for lag in seasonal_lags:
            ax.axvline(x=lag, color='red', linestyle='--', alpha=0.6)
    # différenciation saisonnière (saisonalités sur X lags provenant de la configuration dans le dictionnaire)
    s1 = dict_compteurs[compteur]["seasonal_order"][3]
    # - l'autocorrelation simple (ACF - basé sur les X précédentes observations) 
    # - l'autocorrelation partielle (PACF basée sur les moyennes mobiles de période P)
    # Ces éléments permettent de pré-identifier les hypoerparamètres de SARIMA
    # - d/D selon si les ACF et PACF tendent vers 0 ou stoppent net (ARMA si les deux tendent vers 0)
    # - p/q grâce au 1er lag ACF/PACF entrant dans la zone statistiquement peu probable
    # - P/Q grâce au 1er pic de lag cyclique (à motif saisonnier) ACF/PACF entrant dans la première zone statistiquement peu probable
    fig, (
        (ax1, ax2), 
        (ax3, ax4), 
    ) = plt.subplots(2, 2, figsize=(20,20))
    max_lag1 = s1*7
    seasonal_lags1 = [k * s1 for k in range(1, (max_lag1 // s1) + 1)]
    # 
    y = y_train
    # Différenciation sur la base de la saisonnalité définie pour observer les ordres p, q et P et Q sur la base de s1 lags (saison définie)
    y_s1 = y.diff(s1).dropna()
    plot_pacf(y, lags = max_lag1, ax=ax1)
    ax1.set_title(f'PACF sans différenciation (pour déterminer p)')
    plot_acf(y, lags = max_lag1, ax=ax2)
    ax2.set_title(f'ACF sans différenciation (pour déterminer q)')
    plot_pacf(y_s1, lags = max_lag1, ax=ax3)
    ajouter_lignes_saisonnieres(ax3, seasonal_lags1)
    ax3.set_title(f'PACF avec 1ère différenciation {s1} lags (heures) (pour déterminer P)')
    plot_acf(y_s1, lags = max_lag1, ax=ax4)
    ajouter_lignes_saisonnieres(ax4, seasonal_lags1)
    ax4.set_title(f'ACF avec 1ère différenciation {s1} lags (heures) (pour déterminer Q)')
    plt.show()

In [ ]:
models_sarimax_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    if compteur_id not in dict_compteurs:
        continue
    
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values(by=timestamp_col)
    sub_range = dict_compteurs[compteur_id]["sub_range"]
    df_compteur_sub = df_compteur[sub_range.start:sub_range.stop]
    X_train, X_test, y_train, y_test, dict_missing_range = \
        dfps.train_test_split_time_aware_sarimax(
            df_compteur_sub,
            timestamp_col=timestamp_col,
            target_col=target_col,
            test_size=0.2,
        )

    print(f"ACF et PACF pour {compteur_id}")
    affichage_pacf_acf()

    # pipeline + fit
    order = dict_compteurs[compteur_id]["order"]
    seasonal_order = dict_compteurs[compteur_id]["seasonal_order"]
    use_exo = dict_compteurs[compteur_id]["use_exo"]
    pipe_sarimax = Pipeline(
        steps= [
            ('prep', tr_columns), 
            ('reg',mps.SARIMAXWrapper(order=order, seasonal_order=seasonal_order, use_exo=use_exo))
        ]
    )
    pipe_sarimax.fit(X_train, y_train)

    # predictions
    y_train_pred = pipe_sarimax.predict(X_train)
    y_test_pred = pipe_sarimax.predict(X_test)

    # stockage des resultats
    models_sarimax_results[compteur_id] = {
        "pipe":pipe_sarimax,
        "X_train":X_train,
        "X_train_dates":X_train.reset_index()[detect_datetime_columns(X_train.reset_index())],
        "X_test":X_test,
        "X_test_dates":X_test.reset_index()[detect_datetime_columns(X_test.reset_index())],
        "y_train":y_train,
        "y_train_pred":y_train_pred,
        "y_test":y_test,
        "y_test_pred":y_test_pred,
    }

In [ ]:
def save_sarimax_models(models_results: dict, dict_compteurs: dict, save_dir: str = "."):
    """
    Save SARIMAX models using naming based on hyperparameters and metadata.

    Args:
        models_results: Dict of fitted model results by compteur_id.
        dict_compteurs: Dict with compteur_id as key and metadata (order, name, sub_range).
        save_dir: Directory where to save the files (default: current folder).
    """
    for compteur_id, meta in dict_compteurs.items():
        if compteur_id not in models_results:
            print(f"Skipping {compteur_id}: no result found.")
            continue

        order = meta["order"]
        exo = "exo_" if meta["use_exo"] else ""
        seasonal = meta["seasonal_order"]
        name = meta["name"].replace(" ", "-")
        start = meta["sub_range"].start
        stop = meta["sub_range"].stop - 1

        filename = (
            f"sarimax_results_{exo}"
            f"{order[0]}-{order[1]}-{order[2]}_"
            f"{seasonal[0]}-{seasonal[1]}-{seasonal[2]}-{seasonal[3]}_"
            f"{name}_{start}-{stop}.joblib"
        )
        filepath = f"{save_dir}/{filename}"
        joblib.dump(models_results[compteur_id], filepath)
        print(f"Saved: {filepath}")

In [ ]:
# on enregistre la mémoire avec joblib afin de permettre la resitution rapide (Attention il faut environ 1,5Go par enregistrement différent)
save_sarimax_models(models_sarimax_results, dict_compteurs=dict_compteurs)

# 4 Performance analysis and visualisation

In [ ]:
# dict_compteurs = {
#     ('Totem 73 boulevard de Sébastopol', 'S-N'): {
#         "name": "Sebastopol S-N",
#         "sub_range": range(1500, 5001),
#         "order": (6, 1, 3),
#         "seasonal_order": (1, 1, 1, 24),
#         "use_exo": True,
#     }
# }
# timestamp_col = "date_et_heure_de_comptage_local"
# target_col = "comptage_horaire"

In [ ]:
# On construit une liste en rechargeant les enregisrements de résultats depuis le disque
# cle_compteur = ('Totem 73 boulevard de Sébastopol', 'S-N')
models_sarimax_best = joblib.load("sarimax_results_exo_6-1-3_1-1-1-24_Sebastopol-S-N_1500-5000.joblib")
# il faut restituer les clés (nom_du_site_de_comptage, orientation_compteur), seule information non sauvegardée
liste_model_results = {
    "SARIMAX Best (sarimax_results_6-1-3_1-1-1-24_Sebastopol-S-N_1500-5000.joblib)": {
        ('Totem 73 boulevard de Sébastopol', 'S-N'):models_sarimax_best,
    },
}

In [ ]:
# Afficher une modélisation de compteur spécifique
for compteur, (model_name, model_results) in itertools.product(dict_compteurs, liste_model_results.items()):
    print("\n", "*"*80, "\n", f"Modèle {model_name} :","\n", "*"*80, "\n")
    dates = model_results[compteur]["X_test_dates"]["date_et_heure_de_comptage_local"]
    X_test_dates = model_results[compteur]["X_test_dates"]
    y_test = model_results[compteur]["y_test"]
    y_train = model_results[compteur]["y_train"]
    y_train_pred = model_results[compteur]["y_train_pred"]
    y_test_pred = model_results[compteur]["y_test_pred"]
    periode_limite = (
        dates[len(dates)-200],
        dates[len(dates)-1]
    )

    # Affichage des metrique train et test
    model_train_metrics = mps.compute_metrics(
        y_train,
        y_train_pred
    )
    model_test_metrics = mps.compute_metrics(
        y_test,
        y_test_pred
    )
    print("Metriques du modèle (Train):",model_train_metrics)
    print("Metriques du modèle (Test):",model_test_metrics)

    # projection des predictions de test dans le temps
    fig_pred = mps.plot_predictions(
        str(compteur),
        X_test_dates, 
        y_test, 
        y_test_pred, 
        periode_limite=periode_limite
    )
    plt.show()

    # projection des résidus et calcul du coefficient de dérive dans le temps
    fig1_res, fig2_res, model_res_coeff = mps.compute_residuals_plot(
        str(compteur),
        X_test_dates, 
        y_test, 
        y_test_pred.values, 
        periode_limite=periode_limite
    )
    print("Pente de la droite de régression des résidus dans le temps (dérive) :", model_res_coeff)
    plt.show()